# 04 Publication Tables And Figures

**Purpose.** Consolidate the compact, paper-facing outputs from Notebooks
02a-02g and 03 into one explicit publication inventory and a small final table
set. This notebook does not rerun model development and does not read
exploratory caches.

Every final table is exported as CSV, Markdown, and LaTeX. Missing upstream
artifacts are displayed and treated as execution errors rather than silently
omitted.

**Inputs:** compact tables, figures, and manifests from 02a-02g and 03.  
**Outputs:** seven final table families in three formats, an upstream inventory,
and a reproducibility manifest.  
**Expected runtime:** under one minute after the upstream notebooks have run.

## 1. Imports And Output Contract

Only compact tables, figures, and manifests from the repeatable notebook
workflow are declared below. Paths are relative to the article folder and are
portable across laptops.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    artifact_inventory,
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    write_manifest,
    write_table_formats,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SLUG = "04_publication_tables_figures"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)

## 2. Upstream Artifact Inventory

The table registry selects headline regime, ablation, weight, ML, final
evaluation, localisation, and Gamma forecast results. The figure registry
checks every paper-facing figure generated by 02a-02g and 03. Notebook 02b has
no required plot, so its compact feature-quality table is included in the
inventory as its hand-off artifact.

In [ ]:
SOURCE_TABLES = {
    "table01_m9_training_regimes": PATHS.tables
    / "02c_m9_pbm_training_regimes"
    / "table01_regime_headline_metrics.csv",
    "table02_m9_ablation_by_feature_count": PATHS.tables
    / "02d_m9_pbm_feature_ablation"
    / "table01_best_by_feature_count.csv",
    "table03_m9_weight_optimisation": PATHS.tables
    / "02e_m9_pbm_weight_optimisation"
    / "table01_equal_vs_grid_vs_random.csv",
    "table04_m9_physical_vs_ml": PATHS.tables
    / "02f_m9_pbm_ml_comparison"
    / "table01_physical_vs_ml.csv",
    "table05_m9_final_evaluation": PATHS.tables
    / "02g_m9_pbm_final_evaluation"
    / "table01_final_headline_metrics.csv",
    "table06_m9_localisation_and_energy": PATHS.tables
    / "02g_m9_pbm_final_evaluation"
    / "table03_localisation_and_energy.csv",
    "table07_gamma_forecast_impact": PATHS.tables
    / "03_gamma_forecast_impact"
    / "table02_gamma_forecast_impact.csv",
}

UPSTREAM_FIGURES = {
    "02a_method_example": PATHS.figures
    / "02a_m9_pbm_method_example"
    / "fig01_m9_pbm_alpha_F_2024-02-17.png",
    "02c_regime_metrics": PATHS.figures
    / "02c_m9_pbm_training_regimes"
    / "fig01_regime_precision_recall_f1.png",
    "02c_regime_thresholds": PATHS.figures
    / "02c_m9_pbm_training_regimes"
    / "fig02_thresholds_by_heldout_substation.png",
    "02d_ablation_feature_count": PATHS.figures
    / "02d_m9_pbm_feature_ablation"
    / "fig01_f1_by_feature_count.png",
    "02d_ablation_top_subsets": PATHS.figures
    / "02d_m9_pbm_feature_ablation"
    / "fig02_top_subset_performance.png",
    "02d_ablation_feature_evidence": PATHS.figures
    / "02d_m9_pbm_feature_ablation"
    / "fig03_feature_frequency_and_marginal_effect.png",
    "02e_weight_simplex": PATHS.figures
    / "02e_m9_pbm_weight_optimisation"
    / "fig01_weight_simplex_performance.png",
    "02e_weight_stability": PATHS.figures
    / "02e_m9_pbm_weight_optimisation"
    / "fig02_selected_weights_by_fold.png",
    "02f_physical_vs_ml": PATHS.figures
    / "02f_m9_pbm_ml_comparison"
    / "fig01_physical_vs_ml_precision_recall_f1.png",
    "02g_confusion": PATHS.figures
    / "02g_m9_pbm_final_evaluation"
    / "fig01_final_confusion_matrices.png",
    "02g_window_iou": PATHS.figures
    / "02g_m9_pbm_final_evaluation"
    / "fig02_window_iou_distribution.png",
    "02g_energy": PATHS.figures
    / "02g_m9_pbm_final_evaluation"
    / "fig03_energy_metric_summary.png",
    "02g_review_burden": PATHS.figures
    / "02g_m9_pbm_final_evaluation"
    / "fig04_auto_accept_manual_review_and_errors.png",
    "02g_coverage_scores": PATHS.figures
    / "02g_m9_pbm_final_evaluation"
    / "fig05_auto_accept_precision_recall_f1.png",
    "03_gamma_week": PATHS.figures
    / "03_gamma_forecast_impact"
    / "fig01_gamma_raw_m9_manual_example_week.png",
    "03_gamma_data_error": PATHS.figures
    / "03_gamma_forecast_impact"
    / "fig02_gamma_data_error_rmse.png",
    "03_gamma_forecast_rmse": PATHS.figures
    / "03_gamma_forecast_impact"
    / "fig03_gamma_forecast_rmse.png",
    "03_gamma_residuals": PATHS.figures
    / "03_gamma_forecast_impact"
    / "fig04_gamma_forecast_residuals.png",
}

SUPPORTING_OUTPUTS = {
    "02b_feature_quality": PATHS.tables
    / "02b_m9_pbm_candidate_features"
    / "table02_feature_quality_summary.csv",
    **{
        f"manifest_{name}": PATHS.manifests / f"{name}.json"
        for name in [
            "02a_m9_pbm_method_example",
            "02b_m9_pbm_candidate_features",
            "02c_m9_pbm_training_regimes",
            "02d_m9_pbm_feature_ablation",
            "02e_m9_pbm_weight_optimisation",
            "02f_m9_pbm_ml_comparison",
            "02g_m9_pbm_final_evaluation",
            "03_gamma_forecast_impact",
        ]
    },
}

upstream = {**SOURCE_TABLES, **UPSTREAM_FIGURES, **SUPPORTING_OUTPUTS}
upstream_inventory = artifact_inventory(upstream, relative_to=PATHS.article)
display(upstream_inventory)
assert upstream_inventory["exists"].all(), "Required upstream outputs are missing."
assert upstream_inventory["bytes"].gt(0).all(), "An upstream output is empty."

## 3. Export Final Paper Tables

The exported values are copied from compact upstream tables without
re-estimating or manually transcribing metrics. CSV supports programmatic use,
Markdown supports rapid review, and LaTeX supports manuscript drafting.

In [ ]:
exported_paths = []
table_audit_rows = []
for final_name, source_path in SOURCE_TABLES.items():
    table = pd.read_csv(source_path)
    paths = write_table_formats(table, OUTPUT_DIRS["tables"] / final_name)
    exported_paths.extend(paths)
    table_audit_rows.append(
        {
            "final_table": final_name,
            "source": str(source_path.relative_to(PATHS.article)).replace("\\", "/"),
            "rows": len(table),
            "columns": len(table.columns),
        }
    )

table_audit = pd.DataFrame(table_audit_rows)
inventory_paths = write_table_formats(
    upstream_inventory,
    OUTPUT_DIRS["tables"] / "table00_upstream_artifact_inventory",
)
exported_paths.extend(inventory_paths)
display(table_audit)

## 4. Final Checks And Manifest

This final audit verifies all three formats for every table and records the
upstream dependency graph. Figures remain in their originating notebook
folders so the inventory has one authoritative path for each image.

In [ ]:
final_inventory = artifact_inventory(
    {path.name + "_" + path.parent.name: path for path in exported_paths},
    relative_to=PATHS.article,
)
assert final_inventory["exists"].all() and final_inventory["bytes"].gt(0).all()

manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=list(upstream.values()),
    outputs=exported_paths,
    row_counts={
        "upstream_artifacts": len(upstream_inventory),
        "final_table_families": len(SOURCE_TABLES),
        "final_table_files": len(exported_paths),
        "upstream_figures": len(UPSTREAM_FIGURES),
    },
)
manifest.update(
    {
        "status": "publication_ready",
        "source_notebooks": ["02a", "02b", "02c", "02d", "02e", "02f", "02g", "03"],
        "missing_upstream_outputs": 0,
        "figure_paths": {
            name: str(path.relative_to(PATHS.article)).replace("\\", "/")
            for name, path in UPSTREAM_FIGURES.items()
        },
    }
)
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)
display(final_inventory)
print(
    f"Wrote {len(SOURCE_TABLES)} final table families and checked "
    f"{len(UPSTREAM_FIGURES)} figures."
)
print(f"Manifest: {MANIFEST_PATH.relative_to(PATHS.article)}")